# Создание витрины resort_features: одна строка = один курорт

## Подготовка окружения

In [1]:
import pandas as pd
import numpy as np

# Назначаем путь-константу к файлам
base = '../data/'

## Присоединение таблиц

### Таблица 1 - Справочник курортов - 32 курорта
* Справочник из 32 горнолыжных курортов - основа исходного пакета данных (сентябрь 2026, снапшот 14.09): названия, регион, координаты и Wikidata ID собраны через MediaWiki API (Википедия/Викиданные).
* Качество вики-точек было неравномерным (26 из 32 - прикидка или прокси), поэтому справочник здесь используется только как скелет: идентификаторы, названия, регионы.
* Координаты в этой таблице - до проверки; итоговые точки приезжают из журнала проверки (таблица 2).

In [2]:
# Сохраняем таблицу с курортами в датафрейм mart
mart = pd.read_csv(base + 'prepared/resorts.csv') 

# Проверяем количество строк и уникальных идентификаторов 
print(len(mart), mart['resort_id'].nunique())
mart.shape

32 32


(32, 10)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 10.

### Таблица 2 - Журнал ручной проверки координат
* Координаты вики-справочника частично указывали не на зону катания, а на адрес офиса или центр города. Ошибка точки искажает сразу три источника: OSM-радиус, климатическую ячейку и расстояние до аэропорта.
* Проверка выполнена вручную 1–2 ноября 2026 по спутниковому слою Яндекс.Карт (зоны катания и подъемники видны на снимке).
* Итог: 21 курорт - точка заменена, 9 - подтверждена, 2 (закрыт, реконструкция) - не проверялись, им оставлены исходные вики-точки.
* По подтвержденным точкам 2–3 ноября пересобраны OSM, климат и аэропорты; статусы работы курортов обновлены 3 ноября.
* Статус coordinate_quality в витрине - итоговый, после ручной проверки (исходные статусы сбора остаются в справочнике resorts.csv).

In [3]:
# Формируем список необходимых колонок и присоединяем журнал проверки координат к датафрейму mart
status_cols = ['resort_id', 'operating', 'status_note',
               'latitude после проверки', 'longitude после проверки',
               'coords_status']   # + итог ручной проверки из журнала
mart = mart.merge(pd.read_csv(base + 'prepared/resort_checks.csv')[status_cols],
                  on='resort_id', how='left', validate='one_to_one')

# Заменяем координаты скелета на обновленные координаты после ручной проверки
mart['latitude'] = mart['latitude после проверки'].fillna(mart['latitude'])
mart['longitude'] = mart['longitude после проверки'].fillna(mart['longitude'])

# Финальный статус координат: итог ручной проверки главнее статуса сбора
mart['coordinate_quality'] = mart['coords_status'].map({
    'replaced': 'official_verified',          # точка заменена на зону катания
    'ok': 'checked_ok',                       # исходная точка подтверждена
    'not_checked': 'manual_review_required',  # точки не проверялись: Кола закрыта, Аджигардак в реконструкции
})

# Удаляем лишние отработанные колонки
mart = mart.drop(columns=['latitude после проверки', 'longitude после проверки',
                          'coords_status'])

# Проверяем корректность данных
print(len(mart), mart['resort_id'].nunique())
print(mart['coordinate_quality'].value_counts())   # ждём 21 / 9 / 2
mart.shape

32 32
coordinate_quality
official_verified         21
checked_ok                 9
manual_review_required     2
Name: count, dtype: int64


(32, 12)

#### Вывод:
* Координаты: 21 курорт - заменены по результатам ручной проверки, 9 - подтверждены без изменений, 2 (закрыт/реконструкция) - не проверялись, остались исходные координаты.
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 12 (добавились operating и status_note; отработанные пары координат удалены).

### Таблица 3 - Инфраструктура из OSM - трассы, подъемники, жилье, еда, достопримечательности, транспорт
* Признаки собраны из OpenStreetMap (Overpass API) в радиусе 6 км вокруг итоговой точки курорта, съем живой базы 2–3 ноября 2026.
* Радиус один для всех - курорты сравнимы.
* Оговорка: OSM считает сегменты трасс, а не маршруты - у Розы Хутор 334 сегмента против 71 официальной трассы; зато суммарная длина по геометрии (159,4 км) сходится с официальными ~160 км, что валидирует слой.
* Для рейтинга OSM-счетчики - "насыщенность зоны катания" (6 признаков ТЗ: трассы, подъемники, жилье, еда, достопримечательности, транспорт); официальные числа трасс - в таблице 7.
* Нулевые значения счетчиков - честные, не пропуски: например, у Гладенькой transport_objects_6km = 0 - в радиусе 6 км не размечено ни одной остановки; для удаленных курортов это похоже на правду, но помним ограничение ТЗ - полнота OpenStreetMap различается между регионами.

In [4]:
# Формируем список колонок-признаков и присоединяем таблицу с инфраструктурой к датафрейму mart
osm_cols = ['resort_id', 'piste_total_km_6km', 'lift_lines_6km',
            'lodging_objects_6km', 'food_objects_6km',
            'attraction_objects_6km', 'transport_objects_6km',
            'osm_overlap_warning']   # справочная: для флагов качества
mart = mart.merge(pd.read_csv(base + 'prepared/resort_osm.csv')[osm_cols],
                  on='resort_id', how='left', validate='one_to_one')

print(len(mart), mart['resort_id'].nunique())   # должно сохраниться 32 32
print(mart['lift_lines_6km'].isna().sum())             # должно быть 0
print(mart['osm_overlap_warning'].notna().sum())       # ожидаем 8 курортов с предупреждением
mart.shape

32 32
0
8


(32, 19)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 19 (добавилось 6 колонок-признаков для составления рейтинга + оверлап для флага качества).
* Проверка показала, что пропущенных значений у колонок-признаков нет.

### Таблица 4 - Климатические показатели - снежные дни, морозные дни, снегопад и зимняя температура
* Зимние показатели рассчитаны по архиву Open-Meteo (Historical API), модель ERA5 seamless, период 1 ноября 2021 - 30 апреля 2026, докачка выполнена 3 ноября 2026 по 32 курортам.
* Отступление от ТЗ: модель era5_land у провайдера отдает пустой снегопад, поэтому взята seamless; снег дополнительно перекрестно сверен с независимым реанализом NASA POWER (слой от 02.11, все 32 курорта).
* Колонка winter_mean_temperature_c_honest - контрольный пересчет зимней средней.
* Ограничение ТЗ: реанализ сглаживает перепады высот и не заменяет метеостанцию на склоне.

In [5]:
# Формируем список колонок-признаков и присоединяем таблицу к датафрейму mart
climate_cols = ['resort_id', 'avg_snow_days_per_season', 'avg_freezing_days_per_season',
                'avg_snowfall_cm_per_season', 'winter_mean_temperature_c_honest']
mart = mart.merge(pd.read_csv(base + 'prepared/resort_climate_features.csv')[climate_cols],
                  on='resort_id', how='left', validate='one_to_one')

print(len(mart), mart['resort_id'].nunique()) 
print(mart['avg_snow_days_per_season'].isna().sum())             # должно быть 0
mart.shape 

32 32
0


(32, 23)

#### Вывод:
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 23 (добавилось 3 признака ТЗ + зимняя температура для сценария рейтинга).
* Проверка показала, что пропущенных значений у колонок-признаков нет.

### Таблица 5 - Транспорт и доступность - код и расстояние до аэропорта + название и расстояние ближайшей ЖД-станции и остановочного пункта
* Расстояние до аэропорта: ближайший аэропорт по базе OurAirports, расстояние по прямой от итоговой точки.
* Пересчитано 2–3 ноября 2026 (после замены точек у 14 курортов расстояние изменилось более чем на 1 км - объяснимо сдвигом координат).
* ЖД: ближайшая станция и остановочный пункт из OSM (радиус 50 км), досбор и чистка 3 ноября.
* Ограничение ТЗ: расстояние по прямой не равно времени в дороге.
* В критериях рейтинга ЖД нет - колонки справочные и для сценариев (например, «семья на электричке»).
* Аэропорт: ближайший плановый ищется по всей России без ограничения радиуса, поэтому пропуск невозможен.

In [6]:
airport_cols = ['resort_id', 'nearest_airport_iata', 'nearest_airport_distance_km']
mart = mart.merge(pd.read_csv(base + 'prepared/resort_airport.csv')[airport_cols],
                  on='resort_id', how='left', validate='one_to_one')

railway_cols = ['resort_id',
                'nearest_railway_station_name', 'nearest_railway_station_distance_km',
                'nearest_railway_halt_name', 'nearest_railway_halt_distance_km']
mart = mart.merge(pd.read_csv(base + 'prepared/resort_osm.csv')[railway_cols],
                  on='resort_id', how='left', validate='one_to_one')

print(mart['nearest_railway_station_distance_km'].isna().sum())  
print(mart['nearest_railway_halt_distance_km'].isna().sum())    

print(len(mart), mart['resort_id'].nunique())   # ожидаем 32 32
mart.shape 

8
10
32 32


(32, 29)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 29 (добавилось 6: код и расстояние до аэропорта, название и расстояние ближайшей ЖД-станции и остановочного пункта).

### Таблица 6 - Трассы и опции с официальных сайтов - количество и сложность трасс, фрирайд, тюбинг
* Данные собраны вручную с официальных сайтов курортов 2–3 ноября 2026 (29 из 32: у закрытого и реконструирующихся курортов данных нет).
* Это первоисточник для рейтинга и дашборда: готовые маршруты понятны пользователю и проверяемы по ссылке, в отличие от OSM-сегментов (таблица 3).
* Перепад высот и длина максимальной трассы - сценарные признаки.
* Фрирайд и тюбинг - бинарные признаки для сегментных сценариев.

In [7]:
trails_cols = ['resort_id', 'len_total_km', 'len_max_km', 'drop_m', 'trails_total',
               'green', 'red', 'black', 'blue', 'freeride', 'tubing']
mart = mart.merge(pd.read_csv(base + 'prepared/resort_trails_options.csv')[trails_cols],
                  on='resort_id', how='left', validate='one_to_one')

print(len(mart), mart['resort_id'].nunique())  # ожидаем 32 32
mart.shape 

32 32


(32, 39)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 39 (добавилось 10 колонок с характеристиками трасс).

### Таблица 7 - Цены - ручной сбор цен на 25/26 и 26/27 сезоны
* Цены собраны вручную с официальных сайтов 3 ноября 2026, 29 из 32 курортов (у неработающих курортов цен нет).
* Смешанные сезоны: у 13 курортов новые цены 2026/27 (почти все акционные и абонементные), у 16 - последняя известная цена 2025/26.
* Признак плавающий: используется только для порядка «очень дорого ↔ очень бюджетно», не как точный прайс.
* Нулевая минимальная цена детского тарифа означает бесплатный проезд для детей - заявлено на сайтах при сборе; возрастная градация не хранится.

In [8]:
prices_cols = ['resort_id', 'adult_price_min_rub', 'adult_price_max_rub', 'child_price_min_rub', 'child_price_max_rub']
mart = mart.merge(pd.read_csv(base + 'manual/prices_season_2026_27.csv')[prices_cols],
                  on='resort_id', how='left', validate='one_to_one')

# кто без цены - с причиной: витрина сама объясняет свои дыры
print(mart.loc[mart['adult_price_min_rub'].isna(),
         ['resort_id', 'name_ru', 'operating', 'status_note']])

print(len(mart), mart['resort_id'].nunique())   
mart.shape   

   resort_id        name_ru       operating  \
11      R012       Телецкий  reconstruction   
14      R015  КолаСпортЛэнд          closed   
22      R023     Аджигардак  reconstruction   

                                          status_note  
11  временно не работают: реконструкция трасс до 2...  
14  Яндекс-карты: не работает; офиц. сайт не обнов...  
22  офиц. сообщество: реконструкция продолжается (...  
32 32


(32, 43)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 43 (добавилось 4 колонок с ценами).
* 3 курорта не имеют цену: Телецкий и Аджигардак - реконструкция, КолаСпортЛэнд - закрыт.

### Таблица 8 - Региональный спрос - поездки и ночевки за 2025 год
* Поездки (ЕМИСС 62157) и ночевки (ЕМИСС 62489) по регионам за 2022–2025: выгрузки из исходного пакета, ночевки за 2022 добраны вручную с fedstat.ru 01.11.2026.
* Таблица приведена к единому формату "регион + год".
* Аномальная динамика помечена флагом growth_needs_review.
* Спрос известен только на уровне региона, поэтому все курорты одного региона получают одинаковые значения.
* Региональный поток не показывает посещаемость отдельного курорта.

In [9]:
regions = pd.read_csv(base + 'prepared/region_year_metrics.csv')
print(len(regions), sorted(regions['year'].unique()))   # 80 строк, годы 2022–2025

# Формируем данные за 2024 и 2025 год
t25 = regions[regions['year'] == 2025][['region', 'trips', 'nights']]
t24 = regions[regions['year'] == 2024][['region', 'trips']]

# Переименовываем колонки 
t25 = t25.rename(columns={'trips': 'trips_2025', 'nights': 'nights_2025'})
t24 = t24.rename(columns={'trips': 'trips_2024'})
region_demand = t25.merge(t24, on='region', how='left', validate='one_to_one')

# Рассчитываем метрики
# Рост храним долей
# Длительность = ночевок на одну поездку
region_demand['trips_growth_2025'] = region_demand['trips_2025'] / region_demand['trips_2024'] - 1
region_demand['avg_stay_nights_2025'] = region_demand['nights_2025'] / region_demand['trips_2025']
region_demand.head()

80 [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,region,trips_2025,nights_2025,trips_2024,trips_growth_2025,avg_stay_nights_2025
0,Иркутская область,1456892.0,6190105,1565224.0,-0.069212,4.248843
1,Кабардино-Балкарская Республика,341320.0,2673177,410694.0,-0.168919,7.831879
2,Камчатский край,516216.0,979734,410853.0,0.256449,1.897915
3,Карачаево-Черкесская Республика,1303991.0,2279411,699802.0,0.863371,1.748027
4,Кемеровская область,1266539.0,6089657,1449604.0,-0.126286,4.808109


In [10]:
# Добавляем данные к mart по полю region
demand_cols = ['region', 'trips_2025', 'trips_growth_2025', 'nights_2025', 'avg_stay_nights_2025']
mart = mart.merge(region_demand[demand_cols], on='region',
                  how='left', validate='many_to_one')

print(len(mart), mart['resort_id'].nunique())  
print(mart['trips_2025'].isna().sum())                
mart.shape                                      

32 32
0


(32, 47)

#### Вывод
* Строки: 32, уникальных resort_id - 32, размножения нет.
* Колонки: 47 (добавилось 4 колонки с региональным спросом, включая справочную nights_2025 - она не признак рейтинга, пригодится для профиля региона на дашборде).
* Проверка показала: нет курортов без trips_2025.

## Выгрузка витрины

### Удаление служебных колонок

In [11]:
print(mart.shape)

mart = mart.drop(columns=['coordinate_source_title', 'coordinate_source_url',
                          'wikidata_id', 'needs_manual_check'])
print(mart.shape)

(32, 47)
(32, 43)


#### Вывод
* Удалено 4 колонки.

### Обязательная проверка данных

In [12]:
# Проверка согласно ТЗ: "resort_scores содержит 32 строки и 32 уникальных resort_id"
print('строк / уникальных id:', len(mart), mart['resort_id'].nunique())  # 32 / 32

# Проверка согласно ТЗ: "поездки и ночевки положительны"
print('неположительных trips/nights:', ((mart['trips_2025'] <= 0) | (mart['nights_2025'] <= 0)).sum())

# Проверка согласно ТЗ: "IATA заполнен, расстояние до аэропорта неотрицательно"
print('пустых IATA:', mart['nearest_airport_iata'].isna().sum(),
      '| отрицательных расстояний:', (mart['nearest_airport_distance_km'] < 0).sum())

# Проверка согласно ТЗ: "после регионального соединения нет пропусков в trips_2025
print('курортов без trips_2025:', mart['trips_2025'].isna().sum())       # 0

# Проверка согласно ТЗ: "в итоговых таблицах нет бесконечностей и деления на ноль"
num = mart.select_dtypes('number')               # только числовые колонки
print('бесконечностей:', np.isinf(num).sum().sum())                      # 0

строк / уникальных id: 32 32
неположительных trips/nights: 0
пустых IATA: 0 | отрицательных расстояний: 0
курортов без trips_2025: 0
бесконечностей: 0


#### Вывод
* Все три проверки успешно пройдены.

### Карта пропусков
* status_note - 29 (заметка только у проблемных, у работающих пусто).
* ЖД: станции - 8, остановочные пункты - 10 (горы без железной дороги).
* трассы: общие числа и фрирайд - 2 (закрыт/реконструкция); цвета, максимальная трасса, тюбинг - 2 (те же двое).
* цены - 3 (Телецкий, КолаСпортЛэнд, Аджигардак).
* климат, OSM, аэропорт, спрос - 0.
* osm_overlap_warning - 24 - нет предупреждений.

In [13]:
# сверяем фактическую карту пропусков с ожиданием выше
mart.isna().sum()

resort_id                               0
name_ru                                 0
region                                  0
latitude                                0
longitude                               0
coordinate_quality                      0
operating                               0
status_note                            29
piste_total_km_6km                      0
lift_lines_6km                          0
lodging_objects_6km                     0
food_objects_6km                        0
attraction_objects_6km                  0
transport_objects_6km                   0
osm_overlap_warning                    24
avg_snow_days_per_season                0
avg_freezing_days_per_season            0
avg_snowfall_cm_per_season              0
winter_mean_temperature_c_honest        0
nearest_airport_iata                    0
nearest_airport_distance_km             0
nearest_railway_station_name            8
nearest_railway_station_distance_km     8
nearest_railway_halt_name         

#### Вывод
* Фактическая карта совпала с ожиданием - необъяснимых дыр нет.

### Сохранение витрины

In [14]:
print(mart.shape)

# Сохранение витрины
mart.to_csv(base + 'mart/resort_features.csv', index=False, encoding='utf-8-sig')
print('сохранено в data/mart/resort_features.csv')

(32, 43)
сохранено в data/mart/resort_features.csv


#### Вывод
* Витрина (32 строки и 43 колонки) успешно сохранена! 